# 🚀 Amazing-QR: Tam Otomatik (CSV) QR Oluşturucu

Bu notebook, **Amazing-QR** projesini tamamen CSV (sipariş listesi) odaklı çalıştırmak için optimize edilmiştir. Manuel formlarla uğraşmanıza gerek yoktur.

---

## 🛠️ 1. Kurulum ve Hazırlık

Sistemi başlatmak için bu hücreyi çalıştırın.

In [ ]:
# Projeyi klonla
!git clone https://github.com/orioninsist/amazing-qr.git
%cd amazing-qr

# Gereksinimleri yükle
!pip install -r requirements.txt
!pip install . # amzqr paketini sisteme yükle

# Klasör yapısını hazırla
!mkdir -p inputs/assets
!mkdir -p output

## 📁 2. Sipariş ve Dosya Yükleme

`order.csv` dosyanızı ve varsa logolarınızı/GIF'lerinizi buraya yükleyin.
- `.csv` dosyaları otomatik olarak `inputs/` klasörüne taşınır.
- Görsel dosyaları otomatik olarak `inputs/assets/` klasörüne taşınır.

In [ ]:
from google.colab import files
import os
import shutil

uploaded = files.upload()

for fn in uploaded.keys():
    if fn.endswith('.csv'):
        shutil.move(fn, 'inputs/order.csv')
        print(f"✅ Sipariş dosyası güncellendi: inputs/order.csv")
    elif fn.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
        dest = f'inputs/assets/{fn}'
        shutil.move(fn, dest)
        print(f"🎨 Görsel/GIF yüklendi: {dest}")
    else:
        print(f"❓ Bilinmeyen dosya: {fn} (İşlenmedi)")

## 📊 3. Siparişleri Hazırla ve Önizle

Bu adım, eksik parametreleri (`level`, `version`, `colorized` vb.) otomatik olarak doldurur ve tüm listeyi (`words, version, level, picture, colorized, contrast, brightness, save_name`) gösterir.
- **Otomatik Dolum:** Sadece `words` ve `picture` girseniz bile sistem en iyi ayarları seçer.
- **Önizleme:** Listeyi kontrol edin, her şey hazırsa toplu işlemi başlatın.

In [ ]:
import pandas as pd
import os
import re
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

csv_path = 'inputs/order.csv'

def slugify(value):
    value = re.sub(r'^https?://', '', str(value))
    value = re.sub(r'[^\w\s-]', '_', value).strip().lower()
    return re.sub(r'[-\s]+', '_', value)[:50]

def load_data():
    if not os.path.exists(csv_path):
        return pd.DataFrame(columns=['words', 'version', 'level', 'picture', 'colorized', 'contrast', 'brightness', 'save_name'])
    
    try:
        df = pd.read_csv(csv_path)
    except:
        return pd.DataFrame(columns=['words', 'version', 'level', 'picture', 'colorized', 'contrast', 'brightness', 'save_name'])
        
    # Ensure all columns exist
    cols = ['words', 'version', 'level', 'picture', 'colorized', 'contrast', 'brightness', 'save_name']
    for c in cols:
        if c not in df.columns: df[c] = None
    
    # Fill defaults and handle types
    df['version'] = pd.to_numeric(df['version'], errors='coerce').fillna(1).astype(int)
    df['level'] = df['level'].fillna('H')
    df['contrast'] = pd.to_numeric(df['contrast'], errors='coerce').fillna(1.0).astype(float)
    df['brightness'] = pd.to_numeric(df['brightness'], errors='coerce').fillna(1.0).astype(float)
    
    for i, row in df.iterrows():
        if pd.isna(row['colorized']) or str(row['colorized']).strip() == '' or str(row['colorized']).strip() == 'nan':
            df.at[i, 'colorized'] = True if pd.notna(row.get('picture')) and str(row.get('picture')).strip() != '' and str(row.get('picture')).strip() != 'nan' else False
        else:
            df.at[i, 'colorized'] = str(row['colorized']).lower() == 'true'
            
        if pd.isna(row['save_name']) or str(row['save_name']).strip() == '' or str(row['save_name']).strip() == 'nan':
            ext = '.gif' if str(row.get('picture', '')).lower().endswith('.gif') else '.png'
            slug = slugify(row['words']) or f"qr_{i+1}"
            df.at[i, 'save_name'] = f"{slug}{ext}"
            
    return df[cols]

def save_data(df):
    # Ensure directory exists
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    df.to_csv(csv_path, index=False)

# UI Logic
output = widgets.Output()

def refresh_view():
    with output:
        clear_output()
        df = load_data()
        
        if df.empty:
            display(HTML("<div style='padding: 20px; background: #fff4e5; border-left: 5px solid #ffa117; border-radius: 4px; margin: 10px 0;'>⚠️ Sipariş listesi boş. Lütfen yeni bir sipariş ekleyin veya CSV yükleyin.</div>"))
        else:
            # Table Header with Premium Style
            header_html = \"\"\"
            <div style="display: grid; grid-template-columns: 250px 150px 150px; gap: 10px; background: #2c3e50; color: white; padding: 12px; border-radius: 8px 8px 0 0; font-weight: bold; font-family: sans-serif;">
                <div>URL / Metin</div>
                <div>Görsel (Asset)</div>
                <div>İşlemler</div>
            </div>
            \"\"\"
            display(HTML(header_html))
            
            rows = []
            for i, row in df.iterrows():
                edit_btn = widgets.Button(description="Düzenle", button_style='info', icon='pencil', layout={'width': '85px'})
                delete_btn = widgets.Button(description="Sil", button_style='danger', icon='trash', layout={'width': '65px'})
                
                # Use default arguments in lambda to capture current index
                edit_btn.on_click(lambda b, idx=i: show_editor(idx))
                delete_btn.on_click(lambda b, idx=i: delete_row(idx))
                
                row_widget = widgets.HBox([
                    widgets.Label(str(row['words']), layout={'width': '250px', 'overflow': 'hidden'}),
                    widgets.Label(str(row['picture']) if pd.notna(row['picture']) and str(row['picture']) != 'nan' else "-", layout={'width': '150px'}),
                    widgets.HBox([edit_btn, delete_btn], layout={'gap': '5px'})
                ], layout={'border-bottom': '1px solid #eee', 'padding': '10px', 'align_items': 'center'})
                rows.append(row_widget)
            
            display(widgets.VBox(rows, layout={'border': '1px solid #eee', 'border-top': 'none', 'border-radius': '0 0 8px 8px', 'background': 'white', 'padding': '5px'}))
            
        add_btn = widgets.Button(description="➕ Yeni Sipariş Ekle", button_style='success', layout={'margin': '20px 0', 'width': '200px', 'height': '40px'})
        add_btn.on_click(lambda b: show_editor(None))
        display(add_btn)

def delete_row(idx):
    df = load_data()
    df = df.drop(idx).reset_index(drop=True)
    save_data(df)
    refresh_view()

def show_editor(idx=None):
    df = load_data()
    if idx is not None:
        item = df.iloc[idx]
    else:
        item = {'words': '', 'version': 1, 'level': 'H', 'picture': '', 'colorized': True, 'contrast': 1.0, 'brightness': 1.0, 'save_name': ''}
    
    # Input Widgets
    words_input = widgets.Text(value=str(item['words']), description='<b>URL/Metin:</b>', placeholder='https://...', style={'description_width': '100px'}, layout={'width': '100%'})
    picture_input = widgets.Text(value=str(item['picture']) if pd.notna(item['picture']) and str(item['picture']) != 'nan' else '', description='<b>Görsel Adı:</b>', placeholder='logo.png (Opsiyonel)', style={'description_width': '100px'}, layout={'width': '100%'})
    version_input = widgets.IntSlider(value=int(item['version']), min=1, max=40, step=1, description='Versiyon:', style={'description_width': '100px'})
    level_input = widgets.Dropdown(options=['L', 'M', 'Q', 'H'], value=str(item['level']), description='Hata Payı:', style={'description_width': '100px'})
    colorized_input = widgets.Checkbox(value=bool(item['colorized']), description='Renklendir (Colorized)', style={'description_width': '100px'})
    contrast_input = widgets.FloatSlider(value=float(item['contrast']), min=0.1, max=3.0, step=0.1, description='Kontrast:', style={'description_width': '100px'})
    brightness_input = widgets.FloatSlider(value=float(item['brightness']), min=0.1, max=3.0, step=0.1, description='Parlaklık:', style={'description_width': '100px'})
    save_name_input = widgets.Text(value=str(item['save_name']), description='<b>Kayıt Adı:</b>', placeholder='auto_generated.png', style={'description_width': '100px'}, layout={'width': '100%'})

    save_btn = widgets.Button(description="Kaydet", button_style='primary', icon='check', layout={'width': '120px'})
    cancel_btn = widgets.Button(description="İptal", button_style='', icon='times', layout={'width': '100px'})
    
    def on_save(b):
        new_row = {
            'words': words_input.value,
            'version': version_input.value,
            'level': level_input.value,
            'picture': picture_input.value,
            'colorized': colorized_input.value,
            'contrast': contrast_input.value,
            'brightness': brightness_input.value,
            'save_name': save_name_input.value
        }
        
        current_df = load_data()
        if idx is not None:
            for col, val in new_row.items():
                current_df.at[idx, col] = val
        else:
            current_df = pd.concat([current_df, pd.DataFrame([new_row])], ignore_index=True)
            
        save_data(current_df)
        refresh_view()

    def on_cancel(b):
        refresh_view()

    save_btn.on_click(on_save)
    cancel_btn.on_click(on_cancel)
    
    with output:
        clear_output()
        title = "✍️ Siparişi Düzenle" if idx is not None else "➕ Yeni Sipariş Ekle"
        display(HTML(f"<h3 style='color: #2980b9; margin-bottom: 20px;'>{title}</h3>"))
        
        form = widgets.VBox([
            words_input, 
            picture_input, 
            widgets.HBox([version_input, level_input]),
            colorized_input,
            widgets.HBox([contrast_input, brightness_input]),
            save_name_input,
            widgets.HBox([save_btn, cancel_btn], layout={'margin': '30px 0 10px 0', 'gap': '15px'})
        ], layout={'padding': '30px', 'border': '1px solid #3498db', 'border-radius': '15px', 'background': '#fdfdfd', 'box-shadow': '0 4px 15px rgba(0,0,0,0.05)'})
        display(form)

# Start the UI
display(HTML(\"\"\"
<div style="background: linear-gradient(135deg, #2c3e50 0%, #34495e 100%); padding: 20px; border-radius: 12px; margin-bottom: 25px; box-shadow: 0 10px 30px rgba(0,0,0,0.1);">
    <h2 style="color: white; margin: 0; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; display: flex; align-items: center;">
        <span style="margin-right: 15px;">📋</span> Sipariş Yönetim Paneli
    </h2>
    <p style="color: #bdc3c7; margin: 10px 0 0 0; font-size: 14px;">Tüm siparişlerinizi buradan düzenleyebilir, silebilir veya yeni eklemeler yapabilirsiniz.</p>
</div>
\"\"\"))
display(output)
refresh_view()


## 🚀 4. Toplu İşlemi Başlat

Bu hücreyi çalıştırdığınızda tüm QR kodları otomatik olarak üretilecektir.

In [ ]:
!python batch_process.py

## ✨ 5. Sonuçları Önizle

Üretilen tüm QR kodlarını aşağıda görebilirsiniz.

In [ ]:
import os
from IPython.display import Image, display, HTML

output_dir = 'output'
if os.path.exists(output_dir):
    # Filter for image files and sort them
    files = sorted([f for f in os.listdir(output_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif'))])

    if not files:
        print("❌ Üretilen dosya bulunamadı.")
    else:
        print(f"✅ {len(files)} adet QR kodu görüntülendi.")
        # CSS Grid for a premium look
        html_content = '<div style="display: flex; flex-wrap: wrap; gap: 25px; justify-content: center; padding: 30px; background: #f0f2f5; border-radius: 20px;">'
        for f in files:
            f_path = os.path.join(output_dir, f)
            html_content += f'''
            <div style="text-align: center; background: white; padding: 15px; border-radius: 15px; box-shadow: 0 10px 20px rgba(0,0,0,0.1); width: 230px;">
                <img src="{f_path}" style="width: 200px; height: 200px; object-fit: contain; border-radius: 10px; margin-bottom: 10px;">
                <p style="font-size: 13px; font-weight: bold; color: #333; word-wrap: break-word; margin: 0; font-family: sans-serif;">{f}</p>
            </div>
            '''
        html_content += '</div>'
        display(HTML(html_content))
else:
    print("⚠️ output klasörü bulunamadı.")

## 📥 6. Sonuçları İndir (ZIP)

Üretilen tüm QR kodlarını ve işlem raporunu ZIP olarak indirin.

In [ ]:
import shutil
from google.colab import files
import datetime

now = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_name = f'amazing_qr_results_{now}'

try:
    shutil.make_archive(zip_name, 'zip', 'output')
    files.download(f'{zip_name}.zip')
    print(f"✅ {zip_name}.zip indiriliyor...")
except Exception as e:
    print(f"❌ Hata: {e}")